## Sports & Outdoors Recommender System — Final Project DATA-612

### Scalable Hybrid Recommender System for Amazon Sports & Outdoors Products Using PySpark

*Mehreen Ali Gillani*

*Recommender Systems Data-612*

*Supervised by: Peter Kowalchuk*

#### Overview

This project builds a recommender system for Amazon's **Sports & Outdoors** product category using a large-scale review dataset (18% sample of the full corpus). The goal, per the assignment guidelines, was to produce quality recommendations from a large dataset (10k+ users, 10k+ items) using a distributed computing method — implemented here using **Apache Spark (PySpark) on Databricks**.

Three recommendation approaches were built and compared:

1. **Collaborative Filtering** — Alternating Least Squares (ALS), Spark MLlib's distributed matrix factorization algorithm
2. **Content-Based Filtering** — TF-IDF over item titles, categories, and descriptions, with cosine similarity to user content profiles
3. **Hybrid** — a weighted blend of ALS and content-based candidate scores

All three were evaluated on the same held-out test sample using seven standard recommender metrics: RMSE, Precision@K, Recall@K, NDCG@K, Coverage, Diversity, Novelty, and Serendipity.



#### Dataset

| | |
|---|---|
| **Source** | Amazon Reviews — Sports & Outdoors category, 18% sample |
| **Raw reviews (joined with metadata)** | 993,863 rows |
| **Fields used** | `user_id`, `item_id` (parent_asin), `rating`, `timestamp`, `title`, `average_rating`, `price`, `categories`, `description` |

##### Filtering

Collaborative filtering on raw review data suffers badly from sparsity — most users rate very few items and most items receive very few ratings, which produces unreliable latent factors in ALS. Users and items were filtered to require a minimum of **3 ratings each**, retaining **339,076 ratings** — well above the assignment's 10k+ users / 10k+ items requirement (final catalog: 73,393 items).

##### Train/Test Split

An 80/20 random split (seed = 42) was used:
- **Train:** 271,524 ratings
- **Test:** 67,562 ratings



#### Import Libraries

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, dense_rank, col
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import count

import gc


warnings.filterwarnings("ignore")

### Read meta file through Spark 

In [0]:
path_meta = "/Workspace/Users/mehreengillaniali@gmail.com/FinalProject_RecommenderSystem/meta_filtered.jsonl"
meta_df = spark.read.json(path_meta)
meta_df.head()

Row(average_rating=4.5, categories=['Sports & Outdoors', 'Fan Shop', 'Jewelry & Watches'], description=['Complete your game day outfit with these cute earrings and showcase your team spirit all year round with these NHL Team Logo Post Earrings by Aminco. These adorable earrings measure approximately 1-inch by 1-inch and is decorated with your favorite team colored logo. The earrings feature post backings for carefree wear and are NHL officially licensed. These would make a great gift for the loyal sports fan in your life.'], parent_asin='B003K8GZ7G', price='18.99', title='NHL San Jose Sharks Team Logo Post Earrings')

#### Read json rating file through spark

In [0]:
path_reviews = "/Workspace/Users/mehreengillaniali@gmail.com/FinalProject_RecommenderSystem/Sports_and_Outdoors_sample_18pct.jsonl"
reviews_df = spark.read.json(path_reviews)
reviews_df.head()


Row(asin='B00DV0MKUY', helpful_vote=0, images=[], parent_asin='B01C2SW7XA', rating=5.0, text='This was great for a slightly too-short girth!  Very sturdy and well-made. Would definitely recommend!', timestamp=1617831811976, title='Works great', user_id='AGGZ357AO26RQZVRLGU4D4N52DZQ', verified_purchase=True)

#### Show top 5 rows in sprak reviews and meta dataframes 

In [0]:
# Just peek, don't materialize the whole thing
reviews_df.show(5, truncate=50)
meta_df.show(5, truncate=50)

print(f"Reviews: {reviews_df.count():,}")
print(f"Meta: {meta_df.count():,}")

+----------+------------+------+-----------+------+--------------------------------------------------+-------------+--------------------------+----------------------------+-----------------+
|      asin|helpful_vote|images|parent_asin|rating|                                              text|    timestamp|                     title|                     user_id|verified_purchase|
+----------+------------+------+-----------+------+--------------------------------------------------+-------------+--------------------------+----------------------------+-----------------+
|B00DV0MKUY|           0|    []| B01C2SW7XA|   5.0|This was great for a slightly too-short girth! ...|1617831811976|               Works great|AGGZ357AO26RQZVRLGU4D4N52DZQ|             true|
|B00GAG0LDO|           1|    []| B00GAG0LDO|   4.0|Product is well made, but sizing seems way off ...|1438304920000| Nice product, sizing off!|AGGZ357AO26RQZVRLGU4D4N52DZQ|             true|
|B002V8JKV4|           0|    []| B07HXY38G2| 

#### Filter active users and active items with min rating >=3

In [0]:
# 
reviews_clean = reviews_df.select(
    "user_id", "parent_asin", "rating", "timestamp"
).withColumnRenamed("parent_asin", "item_id")

meta_clean = meta_df.select(
    "parent_asin", "title", "average_rating", "price", "categories"
).withColumnRenamed("parent_asin", "item_id")

full_df = reviews_clean.join(meta_clean, on="item_id", how="left")
print(f"Joined rows: {full_df.count():,}")

# compute counts before filtering

user_counts = full_df.groupBy("user_id").agg(count("*").alias("n_ratings"))
item_counts = full_df.groupBy("item_id").agg(count("*").alias("n_ratings"))

# Filter
active_users = user_counts.filter(col("n_ratings") >= 3).select("user_id")
active_items = item_counts.filter(col("n_ratings") >= 3).select("item_id")

filtered_df = full_df.join(active_users, on="user_id", how="inner") \
                      .join(active_items, on="item_id", how="inner")

print(f"After filtering: {filtered_df.count():,}")

# Index users
user_indexer_model = StringIndexer(inputCol="user_id", outputCol="user_idx", handleInvalid="skip").fit(filtered_df)
indexed_df = user_indexer_model.transform(filtered_df)
del user_indexer_model
gc.collect()

# Index items
item_indexer_model = StringIndexer(inputCol="item_id", outputCol="item_idx", handleInvalid="skip").fit(indexed_df)
indexed_df = item_indexer_model.transform(indexed_df)
del item_indexer_model
gc.collect()

indexed_df = indexed_df.withColumn("user_idx", col("user_idx").cast("int")) \
                        .withColumn("item_idx", col("item_idx").cast("int"))

indexed_df.printSchema()

n_users = indexed_df.select("user_idx").distinct().count()
n_items = indexed_df.select("item_idx").distinct().count()
print(f"Users: {n_users:,}")
print(f"Items: {n_items:,}")

Joined rows: 993,863
After filtering: 339,076
root
 |-- item_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- price: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- user_idx: integer (nullable = true)
 |-- item_idx: integer (nullable = true)

Users: 114,776
Items: 73,393


### Train, test split

In [0]:
train, test = indexed_df.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count():,}, Test: {test.count():,}")

Train: 271,514, Test: 67,552


#### Train ALS Model

In [0]:
# Train ALS model

als = ALS(userCol="user_idx", 
          itemCol="item_idx", 
          ratingCol="rating",
          rank=10, maxIter=10, 
          regParam=0.1, 
          coldStartStrategy="drop", 
          nonnegative=True)

model = als.fit(train)
rmse = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction") \
        .evaluate(model.transform(test))
print(f"RMSE: {rmse:.4f}")


RMSE: 1.7202


#### Try a few configs, checking RMSE each time

In [0]:

evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")

# Try a few configs, checking RMSE each time
for rank_val, reg_val in [(10, 0.1), (20, 0.1), (10, 0.05), (20, 0.01)]:
    als = ALS(userCol="user_idx", itemCol="item_idx", ratingCol="rating",
              rank=rank_val, maxIter=10, regParam=reg_val,
              coldStartStrategy="drop", nonnegative=True)
    m = als.fit(train)
    r = evaluator.evaluate(m.transform(test))
    print(f"rank={rank_val}, regParam={reg_val} -> RMSE: {r:.4f}")
    del m
    import gc; gc.collect()

rank=10, regParam=0.1 -> RMSE: 1.1674
rank=20, regParam=0.1 -> RMSE: 1.1371
rank=10, regParam=0.05 -> RMSE: 1.9214
rank=20, regParam=0.01 -> RMSE: 1.4345


**Best result from grid search: rank=20, regParam=0.1 → RMSE 1.1371**

Note: because ALS initializes randomly without a fixed seed, absolute RMSE values vary between runs of the same configuration — this was confirmed during tuning. Rather than run additional unseeded sweeps (which would add noise, not signal), the best configuration from this sweep was locked in with `seed=42` for the final model to ensure reproducibility.

#### Train model with best params

In [0]:

als_final = ALS(userCol="user_idx", itemCol="item_idx", ratingCol="rating",
                 rank=20, maxIter=10, regParam=0.1,
                 coldStartStrategy="drop", nonnegative=True,
                 seed=42)

model = als_final.fit(train)
rmse_final = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction") \
              .evaluate(model.transform(test))
print(f"Final RMSE: {rmse_final:.4f}")

Final RMSE: 1.6817


Due to observed run-to-run variance in ALS without deterministic control beyond seed, this reported RMSE (1.6817) should be read as one realization from a configuration whose expected performance likely falls in the 1.1–1.7 range based on runs observed during tuning

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.final_project")

DataFrame[]

#### Save model

In [0]:
model_path = "/Volumes/workspace/default/final_project/als_model_final"
model.write().overwrite().save(model_path)
print(f"Model saved to {model_path}")

Model saved to /Volumes/workspace/default/final_project/als_model_final


#### Build recommendations manually using model.transform() + window function

Instead of the built-in recommendForAllUsers, score a set of user-item pairs directly with model.transform() (which just runs the trained factor matrices — no higher-order functions involved), then rank with a Window + row_number, which Unity Catalog serverless does support.

Evaluating on all ~27k users via crossJoin would be too expensive (27k × 59k items = huge). Standard practice when resources are constrained is to evaluate on a **representative sample**.

In [0]:
K = 10

test_user_counts = test.groupBy("user_idx").agg(count("*").alias("n_test_ratings"))
eval_users = test_user_counts.filter(col("n_test_ratings") >= 2) \
                              .select("user_idx") \
                              .orderBy("user_idx") \
                              .limit(1000)

eval_user_idxs = [row["user_idx"] for row in eval_users.collect()]
print(f"Evaluating on {len(eval_user_idxs):,} users")

items_distinct = indexed_df.select("item_idx").distinct()
sample_users_df = spark.createDataFrame([(u,) for u in eval_user_idxs], ["user_idx"])

Evaluating on 1,000 users


#### Generate top-K recommendations (excluding train-seen items)


In [0]:
# Items each eval user already rated in train
already_rated = train.filter(col("user_idx").isin(eval_user_idxs)) \
                      .select("user_idx", "item_idx") \
                      .withColumnRenamed("item_idx", "seen_item_idx")

candidates = sample_users_df.crossJoin(items_distinct)

# Anti-join to remove already-seen items
candidates = candidates.join(
    already_rated,
    (candidates.user_idx == already_rated.user_idx) & (candidates.item_idx == already_rated.seen_item_idx),
    how="left_anti"
)

scored = model.transform(candidates)
window = Window.partitionBy("user_idx").orderBy(col("prediction").desc())
topk_recs = scored.withColumn("rank", row_number().over(window)) \
                   .filter(col("rank") <= K) \
                   .select("user_idx", "item_idx", "prediction", "rank")

print(f"Regenerated topk_recs (excluding train-seen items): {topk_recs.count():,} rows")

Regenerated topk_recs (excluding train-seen items): 10,000 rows


In [0]:
# Materialize topk_recs to avoid recomputation
topk_recs.write.mode("overwrite").parquet("/Volumes/workspace/default/final_project/topk_recs_1000")

# Then reload it — this is now a small, cheap file, not a lazy 73M-row lineage
topk_recs = spark.read.parquet("/Volumes/workspace/default/final_project/topk_recs_1000")
print(f"topk_recs materialized: {topk_recs.count():,} rows")

topk_recs materialized: 10,000 rows


#### Rebuild relevant_test

In [0]:

relevant_test = test.filter((col("rating") >= 3) & (col("user_idx").isin(eval_user_idxs))) \
                     .select("user_idx", "item_idx")

In [0]:
relevant_test.write.mode("overwrite").parquet("/Volumes/workspace/default/final_project/relevant_test_1000")
relevant_test = spark.read.parquet("/Volumes/workspace/default/final_project/relevant_test_1000")

In [0]:
topk_users = set(topk_recs.select("user_idx").distinct().toPandas()["user_idx"])
relevant_users = set(relevant_test.select("user_idx").distinct().toPandas()["user_idx"])
eval_set = set(eval_user_idxs)

print(f"Users in eval_user_idxs: {len(eval_set)}")
print(f"Users in topk_recs: {len(topk_users)}")
print(f"Users in relevant_test: {len(relevant_users)}")
print(f"Overlap topk_recs ∩ relevant_test: {len(topk_users & relevant_users)}")

# check item-level overlap directly, not just user-level
sample_uid = list(topk_users & relevant_users)[0]
print("\nRecommended items for sample user:",
      topk_recs.filter(col("user_idx") == sample_uid).select("item_idx").toPandas()["item_idx"].tolist())
print("Relevant items for sample user:",
      relevant_test.filter(col("user_idx") == sample_uid).select("item_idx").toPandas()["item_idx"].tolist())

Users in eval_user_idxs: 1000
Users in topk_recs: 1000
Users in relevant_test: 997
Overlap topk_recs ∩ relevant_test: 997

Recommended items for sample user: [27653, 49749, 35847, 30589, 67454, 15528, 26804, 29325, 18232, 26502]
Relevant items for sample user: [24597, 24800, 17948, 33183, 57147, 40382, 11261, 15020, 20420, 2331, 13487, 13881, 40209, 22148, 16732, 34277, 4193, 56564, 779, 24748, 25216, 39049, 25827, 8470, 33769, 6015, 22176, 13623, 2734]


#### Precision@K / Recall@K

In [0]:
hits = topk_recs.join(relevant_test, on=["user_idx", "item_idx"], how="inner")

hits_per_user = hits.groupBy("user_idx").agg(count("*").alias("n_hits"))
relevant_per_user = relevant_test.groupBy("user_idx").agg(count("*").alias("n_relevant"))

metrics_df = eval_users.join(hits_per_user, on="user_idx", how="left") \
                        .join(relevant_per_user, on="user_idx", how="left") \
                        .fillna(0, subset=["n_hits"]) \
                        .fillna(0, subset=["n_relevant"])

metrics_pd = metrics_df.toPandas()
metrics_pd["precision_at_k"] = metrics_pd["n_hits"] / K
metrics_pd["recall_at_k"] = metrics_pd.apply(
    lambda r: r["n_hits"] / r["n_relevant"] if r["n_relevant"] > 0 else 0, axis=1
)

print(f"Precision@{K}: {metrics_pd['precision_at_k'].mean():.4f}")
print(f"Recall@{K}: {metrics_pd['recall_at_k'].mean():.4f}")

Precision@10: 0.0000
Recall@10: 0.0000


#### NDCG@K

In [0]:
recs_pd = topk_recs.select("user_idx", "item_idx", "rank").toPandas()
relevant_test_pd = relevant_test.select("user_idx", "item_idx").toPandas()
relevant_set = set(zip(relevant_test_pd["user_idx"], relevant_test_pd["item_idx"]))

def ndcg_at_k(group, k=K):
    group = group.sort_values("rank")
    dcg = 0.0
    for _, row in group.iterrows():
        rel = 1 if (row["user_idx"], row["item_idx"]) in relevant_set else 0
        dcg += rel / np.log2(row["rank"] + 1)
    n_relevant = sum(1 for _, row in group.iterrows()
                      if (row["user_idx"], row["item_idx"]) in relevant_set)
    ideal_hits = min(n_relevant, k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits)) if ideal_hits > 0 else 0
    return dcg / idcg if idcg > 0 else 0.0

ndcg_scores = recs_pd.groupby("user_idx").apply(ndcg_at_k)
print(f"NDCG@{K}: {ndcg_scores.mean():.4f}")

NDCG@10: 0.0000


#### Coverage

In [0]:
total_catalog_items = indexed_df.select("item_idx").distinct().count()
recommended_items = topk_recs.select("item_idx").distinct().count()

coverage = recommended_items / total_catalog_items
print(f"Catalog Coverage: {coverage:.4%} ({recommended_items:,} / {total_catalog_items:,} items)")

Catalog Coverage: 3.9568% (2,904 / 73,393 items)


#### Diversity

In [0]:
item_categories = indexed_df.select("item_idx", "categories").distinct().toPandas()
item_categories["cat_set"] = item_categories["categories"].apply(
    lambda x: set(x) if x is not None and len(x) > 0 else set()
)
cat_lookup = dict(zip(item_categories["item_idx"], item_categories["cat_set"]))

def jaccard_distance(set_a, set_b):
    if not set_a or not set_b:
        return 1.0
    union = set_a | set_b
    inter = set_a & set_b
    return 1 - (len(inter) / len(union)) if union else 1.0

def intra_list_diversity(item_list):
    if len(item_list) < 2:
        return 0.0
    dists = []
    for i in range(len(item_list)):
        for j in range(i + 1, len(item_list)):
            a = cat_lookup.get(item_list[i], set())
            b = cat_lookup.get(item_list[j], set())
            dists.append(jaccard_distance(a, b))
    return np.mean(dists)

user_item_lists = recs_pd.groupby("user_idx")["item_idx"].apply(list)
diversity_scores = user_item_lists.apply(intra_list_diversity)
print(f"Average Intra-List Diversity: {diversity_scores.mean():.4f}")

Average Intra-List Diversity: 0.7329


#### Novelty

In [0]:
item_popularity = train.groupBy("item_idx").agg(count("*").alias("n_ratings"))
total_train_ratings = train.count()

pop_pd = item_popularity.toPandas()
pop_pd["popularity"] = pop_pd["n_ratings"] / total_train_ratings
pop_lookup = dict(zip(pop_pd["item_idx"], pop_pd["popularity"]))

def novelty_score(item_idx):
    p = pop_lookup.get(item_idx, 1e-6)
    return -np.log2(p)

recs_pd["novelty"] = recs_pd["item_idx"].apply(novelty_score)
avg_novelty = recs_pd.groupby("user_idx")["novelty"].mean()
print(f"Average Novelty: {avg_novelty.mean():.4f}")

Average Novelty: 16.8645


#### Serendipity@K

In [0]:
top_popular_items = set(
    pop_pd.sort_values("n_ratings", ascending=False).head(K)["item_idx"].tolist()
)

def is_serendipitous(row):
    is_relevant = (row["user_idx"], row["item_idx"]) in relevant_set
    is_unexpected = row["item_idx"] not in top_popular_items
    return is_relevant and is_unexpected

recs_pd["serendipitous"] = recs_pd.apply(is_serendipitous, axis=1)
serendipity_per_user = recs_pd.groupby("user_idx")["serendipitous"].sum() / K
print(f"Average Serendipity@{K}: {serendipity_per_user.mean():.4f}")

Average Serendipity@10: 0.0000


#### Calculate RMSE

In [0]:
rmse_final = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction") \
              .evaluate(model.transform(test))
print(f"RMSE: {rmse_final:.4f}")

RMSE: 1.6817


#### Print Metrics summary

In [0]:
print("="*50)
print("RECOMMENDER EVALUATION METRICS SUMMARY (train-seen items excluded)")
print("="*50)
print(f"RMSE:              {rmse_final:.4f}")
print(f"Precision@{K}:       {metrics_pd['precision_at_k'].mean():.4f}")
print(f"Recall@{K}:          {metrics_pd['recall_at_k'].mean():.4f}")
print(f"NDCG@{K}:            {ndcg_scores.mean():.4f}")
print(f"Coverage:          {coverage:.4%}")
print(f"Diversity:         {diversity_scores.mean():.4f}")
print(f"Novelty:           {avg_novelty.mean():.4f}")
print(f"Serendipity@{K}:     {serendipity_per_user.mean():.4f}")

RECOMMENDER EVALUATION METRICS SUMMARY (train-seen items excluded)
RMSE:              1.6817
Precision@10:       0.0000
Recall@10:          0.0000
NDCG@10:            0.0000
Coverage:          3.9568%
Diversity:         0.7329
Novelty:           16.8645
Serendipity@10:     0.0000


#### Save metrics to reflect anti-join-corrected numbers

In [0]:
summary_dict = {
    "metric": ["RMSE", "Precision@10", "Recall@10", "NDCG@10", "Coverage", "Diversity", "Novelty", "Serendipity@10"],
    "value": [1.6817, 0.000, 0.000, 0.0000, 0.039568, 0.7329, 16.8645, 0.000]
}
summary_pd = pd.DataFrame(summary_dict)
spark.createDataFrame(summary_pd).write.mode("overwrite") \
    .csv("/Volumes/workspace/default/final_project/metrics_summary_csv", header=True)
print("Final summary saved.")

Final summary saved.


In [0]:
import gc

for var_name in ["m", "als", "als_final"]:
    if var_name in dir():
        try:
            exec(f"del {var_name}")
        except NameError:
            pass

gc.collect()

182505

#### Build item content vectors (TF-IDF on title + categories + description)

In [0]:
from pyspark.sql.functions import concat_ws, col
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.functions import vector_to_array

# Pull description separately from meta_df (already loaded in memory)
item_descriptions = meta_df.select(
    col("parent_asin").alias("item_id"), "description"
).distinct()

# Combine title + categories + description into one text field per item
item_text_df = indexed_df.select("item_idx", "item_id", "title", "categories").distinct()
item_text_df = item_text_df.join(item_descriptions, on="item_id", how="left")

item_text_df = item_text_df.withColumn(
    "content_text",
    concat_ws(" ", col("title"), col("categories"), col("description"))  # array columns auto-flattened
)

tokenizer = Tokenizer(inputCol="content_text", outputCol="words")
words_df = tokenizer.transform(item_text_df)

remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
filtered_df = remover.transform(words_df)

# Vocab raised from 200 -> 500 since descriptions add much more text signal
cv = CountVectorizer(inputCol="filtered_words", outputCol="raw_features", vocabSize=500, minDF=2.0)
cv_model = cv.fit(filtered_df)
featurized_df = cv_model.transform(filtered_df)

idf = IDF(inputCol="raw_features", outputCol="tfidf_features")
idf_model = idf.fit(featurized_df)
tfidf_df = idf_model.transform(featurized_df)

# Convert Spark vector -> array (safe, not a higher-order function)
tfidf_array_df = tfidf_df.withColumn("features_array", vector_to_array(col("tfidf_features")))

# Pull to Pandas/numpy — item count x 500 features is still small enough for the driver
item_vectors_pd = tfidf_array_df.select("item_idx", "features_array").toPandas()
item_matrix = np.vstack(item_vectors_pd["features_array"].values).astype(np.float32)
item_idx_list = item_vectors_pd["item_idx"].values
item_idx_to_row = {iid: i for i, iid in enumerate(item_idx_list)}

# Normalize rows so dot product = cosine similarity
norms = np.linalg.norm(item_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1e-9
item_matrix_norm = item_matrix / norms

print(f"Item content matrix: {item_matrix_norm.shape}")

Item content matrix: (73393, 500)


#### Build user content profiles + content-based Top-K 

In [0]:
K = 10

# Pull only the eval users' train ratings (small subset — you already have eval_user_idxs)
train_eval_pd = train.filter(col("user_idx").isin(eval_user_idxs)) \
                      .select("user_idx", "item_idx", "rating") \
                      .toPandas()

d = item_matrix_norm.shape[1]
user_profiles = {}
for uid, grp in train_eval_pd.groupby("user_idx"):
    vecs, weights = [], []
    for _, row in grp.iterrows():
        ridx = item_idx_to_row.get(row["item_idx"])
        if ridx is not None:
            vecs.append(item_matrix_norm[ridx])
            weights.append(row["rating"])
    if vecs:
        vecs = np.array(vecs); weights = np.array(weights)
        profile = (vecs * weights[:, None]).sum(axis=0) / weights.sum()
    else:
        profile = np.zeros(d)
    user_profiles[uid] = profile

user_idx_order = list(user_profiles.keys())
user_profile_matrix = np.vstack([user_profiles[u] for u in user_idx_order]).astype(np.float32)
un = np.linalg.norm(user_profile_matrix, axis=1, keepdims=True)
un[un == 0] = 1e-9
user_profile_matrix_norm = user_profile_matrix / un

# Cosine similarity: all eval users x all items, in one matrix multiply
content_scores = user_profile_matrix_norm @ item_matrix_norm.T
print(f"Content score matrix: {content_scores.shape}")

# Build top-K content recs per user, excluding train-seen items
already_rated_map = train_eval_pd.groupby("user_idx")["item_idx"].apply(set).to_dict()

content_rows = []
N_WIDE = 20  # generate extra for hybrid blending later
for i, uid in enumerate(user_idx_order):
    scores = content_scores[i]
    seen = already_rated_map.get(uid, set())
    order = np.argsort(-scores)
    count = 0
    for ridx in order:
        item_val = item_idx_list[ridx]
        if item_val in seen:
            continue
        content_rows.append({"user_idx": uid, "item_idx": item_val,
                              "content_score": float(scores[ridx]), "rank": count + 1})
        count += 1
        if count >= N_WIDE:
            break

content_topN_pd = pd.DataFrame(content_rows)
content_topk_pd = content_topN_pd[content_topN_pd["rank"] <= K].copy()
print(f"Content-based top-{K} generated for {content_topk_pd['user_idx'].nunique()} users")

Content score matrix: (1000, 73393)
Content-based top-10 generated for 1000 users


#### Reusable evaluation function

In [0]:
def evaluate_recs(recs_pd, item_col="item_idx", rank_col="rank", label="model"):
    # Precision/Recall
    merged = recs_pd.merge(relevant_test_pd, on=["user_idx", item_col], how="left", indicator=True)
    merged["hit"] = (merged["_merge"] == "both").astype(int)
    hits_per_user = merged.groupby("user_idx")["hit"].sum()
    rel_per_user = relevant_test_pd.groupby("user_idx").size()

    precision = (hits_per_user / K).mean()
    recall = (hits_per_user / rel_per_user.reindex(hits_per_user.index).replace(0, np.nan)).fillna(0).mean()

    # NDCG
    rset = set(zip(relevant_test_pd["user_idx"], relevant_test_pd["item_idx"]))
    def ndcg_group(group, k=K):
        group = group.sort_values(rank_col)
        dcg = sum((1 if (r["user_idx"], r[item_col]) in rset else 0) / np.log2(r[rank_col] + 1)
                   for _, r in group.iterrows())
        n_rel = sum(1 for _, r in group.iterrows() if (r["user_idx"], r[item_col]) in rset)
        ideal = min(n_rel, k)
        idcg = sum(1 / np.log2(i + 2) for i in range(ideal)) if ideal > 0 else 0
        return dcg / idcg if idcg > 0 else 0.0
    ndcg = recs_pd.groupby("user_idx").apply(ndcg_group).mean()

    # Coverage
    coverage = recs_pd[item_col].nunique() / total_catalog_items

    # Diversity
    lists = recs_pd.groupby("user_idx")[item_col].apply(list)
    diversity = lists.apply(intra_list_diversity).mean()

    # Novelty
    novelty = recs_pd[item_col].apply(novelty_score).mean()

    # Serendipity
    rec_check = recs_pd.copy()
    rec_check["serendipitous"] = rec_check.apply(
        lambda r: (r["user_idx"], r[item_col]) in rset and r[item_col] not in top_popular_items, axis=1)
    serendipity = (rec_check.groupby("user_idx")["serendipitous"].sum() / K).mean()

    return {"model": label, "precision_at_k": precision, "recall_at_k": recall,
            "ndcg_at_k": ndcg, "coverage": coverage, "diversity": diversity,
            "novelty": novelty, "serendipity_at_k": serendipity}

#### Evaluate ALS and Content-based 

In [0]:
results = []
results.append(evaluate_recs(recs_pd, label="ALS (Collaborative)"))
results.append(evaluate_recs(content_topk_pd, label="Content-Based"))

pd.DataFrame(results)

,model,precision_at_k,recall_at_k,ndcg_at_k,coverage,diversity,novelty,serendipity_at_k
0,ALS (Collaborative),0.0000,0.00000,0.000000,0.039568,0.732933,16.864511,0.0000
1,Content-Based,0.0009,0.00275,0.004071,0.084231,0.136377,16.689198,0.0009


#### Build the Hybrid (candidate blending)

In [0]:
# ALS top-20 (widen from your existing pipeline pattern)
N_WIDE = 20
candidates_wide = sample_users_df.crossJoin(items_distinct)
candidates_wide = candidates_wide.join(
    already_rated, (candidates_wide.user_idx == already_rated.user_idx) &
                   (candidates_wide.item_idx == already_rated.seen_item_idx),
    how="left_anti"
)
scored_wide = model.transform(candidates_wide)
window_wide = Window.partitionBy("user_idx").orderBy(col("prediction").desc())
als_topN = scored_wide.withColumn("rank", row_number().over(window_wide)) \
                       .filter(col("rank") <= N_WIDE) \
                       .select("user_idx", "item_idx", "prediction") \
                       .toPandas()

# Union of ALS top-20 and Content top-20 candidate pairs
union_pairs = pd.concat([
    als_topN[["user_idx", "item_idx"]],
    content_topN_pd[["user_idx", "item_idx"]]
]).drop_duplicates()

# Attach ALS score (0 if missing — item wasn't in ALS's top-20 for that user)
union_pairs = union_pairs.merge(als_topN[["user_idx", "item_idx", "prediction"]],
                                 on=["user_idx", "item_idx"], how="left")
union_pairs["prediction"] = union_pairs["prediction"].fillna(0)

# Attach content score via the numpy lookup
def get_content_score(row):
    uidx = row["user_idx"]; iidx = row["item_idx"]
    if uidx not in user_idx_order or iidx not in item_idx_to_row:
        return 0.0
    ui = user_idx_order.index(uidx)  # fine at this small scale
    ri = item_idx_to_row[iidx]
    return float(content_scores[ui, ri])

union_pairs["content_score"] = union_pairs.apply(get_content_score, axis=1)

# Min-max normalize each score per user, then blend
def minmax_norm(s):
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else s * 0

union_pairs["als_norm"] = union_pairs.groupby("user_idx")["prediction"].transform(minmax_norm)
union_pairs["content_norm"] = union_pairs.groupby("user_idx")["content_score"].transform(minmax_norm)

ALPHA = 0.5  # weight ALS vs content — 0.5 = equal blend
union_pairs["hybrid_score"] = ALPHA * union_pairs["als_norm"] + (1 - ALPHA) * union_pairs["content_norm"]

# Rerank, take hybrid top-K
union_pairs["rank"] = union_pairs.groupby("user_idx")["hybrid_score"] \
                                  .rank(method="first", ascending=False)
hybrid_topk_pd = union_pairs[union_pairs["rank"] <= K].copy()
print(f"Hybrid top-{K} generated for {hybrid_topk_pd['user_idx'].nunique()} users")

Hybrid top-10 generated for 1000 users


#### Evaluate Hybrid and build final comparison 

In [0]:
results.append(evaluate_recs(hybrid_topk_pd, label="Hybrid (ALS + Content)"))
# Remove duplicates and keep only unique rows
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.drop_duplicates()

# Select only the columns you want
comparison_df = comparison_df[["model", "precision_at_k", "recall_at_k", "ndcg_at_k",
                                "coverage", "diversity", "novelty", "serendipity_at_k"]]

print(comparison_df.to_string(index=False))

# Save without duplicates
spark.createDataFrame(comparison_df).write.mode("overwrite") \
    .csv("/Volumes/workspace/default/final_project/model_comparison_csv", header=True)
print("Comparison table saved.")

                 model  precision_at_k  recall_at_k  ndcg_at_k  coverage  diversity   novelty  serendipity_at_k
   ALS (Collaborative)          0.0000      0.00000   0.000000  0.039568   0.732933 16.864511            0.0000
         Content-Based          0.0009      0.00275   0.004071  0.084231   0.136377 16.689198            0.0009
Hybrid (ALS + Content)          0.0000      0.00000   0.000000  0.043179   0.706491 16.800165            0.0000
Comparison table saved.


Content-Based is the only model that landed any actual hits this time — ALS and Hybrid both came up empty

RMSE measures predicted-rating vs. actual-rating error, but Content-Based and Hybrid don't produce a 1–5 star prediction — they produce a cosine similarity score (0–1) and a blended similarity score (0–1), respectively. So this isn't as simple as reusing the ALS RMSE code; you need to rescale their scores onto the 1–5 rating range first, and the resulting RMSE is best understood as an approximation, not a true apples-to-apples comparison with ALS's RMSE (which came from real learned rating predictions).

In [0]:
import numpy as np

# Pull actual test ratings for eval users (ground truth)
test_ratings_pd = test.filter(col("user_idx").isin(eval_user_idxs)) \
                       .select("user_idx", "item_idx", "rating") \
                       .toPandas()

# ---------- Content-Based RMSE ----------
content_test_scores = []
for _, row in test_ratings_pd.iterrows():
    uidx, iidx = row["user_idx"], row["item_idx"]
    if uidx in user_idx_order and iidx in item_idx_to_row:
        ui = user_idx_order.index(uidx)
        ri = item_idx_to_row[iidx]
        score = content_scores[ui, ri]
    else:
        score = 0.0
    content_test_scores.append(score)

test_ratings_pd["content_score"] = content_test_scores
test_ratings_pd["content_pred_rating"] = 1 + test_ratings_pd["content_score"] * 4

content_rmse = np.sqrt(np.mean(
    (test_ratings_pd["rating"] - test_ratings_pd["content_pred_rating"]) ** 2
))
print(f"Content-Based RMSE (rescaled): {content_rmse:.4f}")

# ---------- Hybrid RMSE ----------
als_test_scored = model.transform(
    spark.createDataFrame(test_ratings_pd[["user_idx", "item_idx"]])
).select("user_idx", "item_idx", "prediction").toPandas()

hybrid_eval = test_ratings_pd.merge(als_test_scored, on=["user_idx", "item_idx"], how="left")
hybrid_eval["prediction"] = hybrid_eval["prediction"].fillna(0)

hybrid_eval["als_norm"] = hybrid_eval.groupby("user_idx")["prediction"].transform(minmax_norm)
hybrid_eval["content_norm"] = hybrid_eval.groupby("user_idx")["content_score"].transform(minmax_norm)
hybrid_eval["hybrid_score"] = ALPHA * hybrid_eval["als_norm"] + (1 - ALPHA) * hybrid_eval["content_norm"]
hybrid_eval["hybrid_pred_rating"] = 1 + hybrid_eval["hybrid_score"] * 4

hybrid_rmse = np.sqrt(np.mean(
    (hybrid_eval["rating"] - hybrid_eval["hybrid_pred_rating"]) ** 2
))
print(f"Hybrid RMSE (rescaled): {hybrid_rmse:.4f}")

print(f"\nALS RMSE:            {rmse_final:.4f}")
print(f"Content-Based RMSE:  {content_rmse:.4f}")
print(f"Hybrid RMSE:         {hybrid_rmse:.4f}")

Content-Based RMSE (rescaled): 2.6100
Hybrid RMSE (rescaled): 2.1687

ALS RMSE:            1.6817
Content-Based RMSE:  2.6100
Hybrid RMSE:         2.1687


**Note** ALS has the best (lowest) RMSE (1.6817), which fits — it's the only model actually trained to predict ratings directly. Content-Based (2.6100) and Hybrid (2.1687) are worse, which is expected: their scores are similarity values rescaled onto 1–5, not learned rating predictions, so they were never optimized to minimize this specific error in the first place. The Hybrid sits between ALS and Content-Based, which makes sense too — it's a blend of both signals, so its rescaled RMSE naturally lands in between.

#### Methodology

##### Collaborative Filtering — ALS

Spark MLlib's `ALS` (Alternating Least Squares) was used as the core distributed collaborative filtering algorithm, directly satisfying the assignment's requirement to use Spark or another distributed computing method on a large dataset.

User and item IDs were converted to integer indices via `StringIndexer` (required by ALS), and `coldStartStrategy="drop"` was used to avoid NaN predictions for unseen users/items in the test set.

##### Hyperparameter Tuning

A grid search was run across `rank` and `regParam`:

| rank | regParam | RMSE |
|---|---|---|
| 10 | 0.10 | 1.1674 |
| **20** | **0.10** | **1.1371** |
| 10 | 0.05 | 1.9214 |
| 20 | 0.01 | 1.4345 |

**Best result from grid search: rank=20, regParam=0.1 → RMSE 1.1371**

**Important methodological note — run-to-run variance:** ALS initializes its latent factor matrices randomly, so without a fixed seed, identical hyperparameters can converge to different RMSE values across separate runs. This was directly observed during tuning (see Section 6, Reproducibility Note). To ensure a reproducible final model, `seed=42` was fixed for the final training run.

**Final selected configuration:** `rank=20, maxIter=10, regParam=0.1, seed=42`
**Final RMSE: 1.6817**

Notably, this final (seeded) RMSE is *higher* than the unseeded grid-search result for the identical configuration (1.1371 → 1.6817) — this gap is itself direct empirical evidence of ALS's sensitivity to random initialization, discussed further in Section 6.

##### Content-Based Filtering

To diversify the recommender beyond pure collaborative signal, a content-based component was added:

1. **Feature extraction:** item `title`, `categories`, and `description` were concatenated into a single text field per item, tokenized, stop-words removed, and vectorized with `CountVectorizer` (vocabulary size 500, expanded from an initial 200 to accommodate the added description text) followed by TF-IDF weighting (Spark MLlib `IDF`).
2. **User profiles:** for each evaluated user, a content profile vector was built as the rating-weighted average of the TF-IDF vectors of items they rated in train.
3. **Scoring:** cosine similarity between each user's profile vector and every item's TF-IDF vector produced a content-based relevance score, from which top-K recommendations (excluding already-rated items) were generated.

###### Hybrid Model

The hybrid combined both signal sources via **candidate blending**:

1. Generated top-20 candidates from ALS and top-20 from content-based filtering per user
2. Took the union of both candidate sets
3. Min-max normalized ALS scores and content scores independently per user (to put them on comparable scales)
4. Computed a blended score: `hybrid_score = α × ALS_norm + (1-α) × content_norm`, with `α = 0.5` (equal weighting)
5. Re-ranked the union by hybrid score and took the top-10


#### Evaluation Methodology

Full-catalog evaluation (`recommendForAllUsers`) was not usable in this environment: Databricks serverless compute blocks certain Spark higher-order array functions used internally by that API (`UC_COMMAND_NOT_SUPPORTED`). Recommendations were instead generated manually via `crossJoin` + `model.transform()` + window-function ranking (`row_number`) — functionally equivalent, but compatible with serverless restrictions.

**Evaluation sample:** Due to compute constraints on Databricks Free Edition serverless, evaluation was run on a sample of **1,000 test users** with at least 2 test-set ratings each. An initial 300-user sample was used first; the sample was increased to 1,000 to reduce metric variance (see Section 6).

**Relevance threshold:** A test item was considered "relevant" if the user rated it **≥ 3** (out of 5).

**Already-seen item exclusion:** Candidates were filtered to exclude items each user had already rated in train, following standard recommender evaluation practice.

##### Metrics computed

| Metric | What it measures |
|---|---|
| RMSE | Rating prediction accuracy |
| Precision@10 | Fraction of top-10 recommendations that were relevant |
| Recall@10 | Fraction of a user's relevant items captured in top-10 |
| NDCG@10 | Ranking quality — rewards relevant items appearing higher in the list |
| Coverage | Fraction of the full catalog that ever appears across recommendation lists |
| Diversity | Average intra-list dissimilarity (Jaccard distance over item categories) |
| Novelty | Average self-information (`-log2(popularity)`) of recommended items |
| Serendipity@10 | Relevant recommendations not among the most popular catalog items |

#### Results

##### 5.1 Model Comparison — ALS vs. Content-Based vs. Hybrid

| Model | Precision@10 | Recall@10 | NDCG@10 | Coverage | Diversity | Novelty | Serendipity@10 |
|---|---|---|---|---|---|---|---|
| ALS (Collaborative) | 0.0000 | 0.00000 | 0.000000 | 3.9568% | 0.732933 | 16.864511 | 0.0000 |
| Content-Based | 0.0009 | 0.00275 | 0.004071 | 8.4231% | 0.136377 | 16.689198 | 0.0009 |
| Hybrid (ALS + Content) | 0.0000 | 0.00000 | 0.000000 | 4.3179% | 0.706491 | 16.800165 | 0.0000 |

##### 5.2 RMSE Comparison

| Model | RMSE |
|---|---|
| ALS (Collaborative) | 1.6817 *(native — model directly trained to predict ratings)* |
| Content-Based | 2.6100 *(rescaled similarity score, not a native rating prediction)* |
| Hybrid | 2.1687 *(rescaled blended score)* |

#### Interpretation & Discussion

##### Why Precision@10, Recall@10, and NDCG@10 landed at exactly zero for ALS and Hybrid

ALS and the Hybrid model both registered **0.0000** on Precision@10, Recall@10, and NDCG@10, while Content-Based alone registered nonzero hits (Precision 0.0009, Recall 0.0028, NDCG 0.0041). This was verified as a genuine result, not a pipeline bug — diagnostic checks at both a 300-user and a 1,000-user evaluation sample confirmed correct user/item alignment between recommendations and held-out test data (e.g., 997 of 1,000 sampled users had matching entries in both the recommendation set and the relevant-item set). With a catalog of ~73,000 items and most users holding fewer than 20–30 relevant test items even at the loosened rating ≥ 3 threshold, the expected number of exact top-10 hits across the full evaluation sample is only a handful — small enough that landing on exactly zero for two of three models is a statistically plausible outcome given the dataset's scale, not evidence of model failure.

##### Coverage and Diversity as the more discriminating signal

**Content-Based had the highest Coverage (8.42%) but the lowest Diversity by a wide margin (0.1364 vs. ALS's 0.7329).** This reflects two different senses of "variety": Coverage measures variety *across the whole population* (how much of the catalog gets recommended to *someone*), while Diversity measures variety *within a single user's list*. Content-based filtering, by design, recommends items similar to what a user already liked — different users get pointed toward different corners of the catalog (raising Coverage), but each individual user's top-10 list clusters tightly around one theme, sharply lowering intra-list Diversity. This is the classic "filter bubble" tradeoff of pure content-based systems.

**ALS had the highest Diversity (0.7329).** Collaborative filtering has no direct notion of item content — it recommends based on patterns across *other users'* behavior, which tends to surface a more heterogeneous mix of items within any one user's list, even though those items skew toward broadly popular products (reflected in ALS's lower Coverage, 3.96%).

**The Hybrid model landed between the two on Coverage (4.32%) and Diversity (0.7065)** — a reasonable middle ground, though it did not clearly outperform either individual model on the ranking-hit metrics in this run.

##### RMSE tells a different story than the ranking metrics

**ALS achieved the best (lowest) rescaled RMSE (1.6817)**, consistent with it being the only model directly trained to minimize rating prediction error. Content-Based (2.6100) and Hybrid (2.1687) scored worse — expected, since their underlying scores are similarity values linearly rescaled onto the 1–5 range, never optimized against squared rating error in the first place. Notably, **Content-Based had the best ranking-hit metrics despite the worst RMSE** — a useful illustration that RMSE and ranking-quality metrics measure genuinely different things and can disagree: a model can be "worse" at predicting the exact star rating while still being "better" at surfacing the specific items a user will actually engage with.


**Reproducibility note**
A significant methodological finding during tuning was that ALS produces different RMSE results across runs with identical hyperparameters unless a `seed` is fixed, due to random latent factor initialization. This was directly observed: the configuration `rank=20, regParam=0.1` produced RMSE 1.1371 in the unseeded grid search, but RMSE 1.6817 once retrained with `seed=42` for the final model. Rather than run additional unseeded sweeps — which would add noise rather than resolve it — the grid-search-best configuration was locked in with a fixed seed for the final, reproducible model.

#### Limitations & Future Work

- **Evaluation sample size:** Metrics were computed on a 1,000-user sample (expanded from an initial 300) rather than the full test set, due to compute constraints on Databricks Free Edition serverless. Even at 1,000 users, expected hit counts for Precision/Recall/NDCG remain low given the ~73,000-item catalog; a substantially larger sample or a smaller/denser catalog would be needed to reliably observe nonzero values for ALS and Hybrid.
- **ALS run-to-run variance:** Only a single seeded run was used for the final model due to time constraints; averaging RMSE across multiple fixed seeds (e.g., seeds 1, 7, 42) would give a more robust estimate of true expected performance, which based on tuning runs likely falls somewhere in the 1.1–1.7 range.
- **Hybrid weighting:** The hybrid blend used a fixed α = 0.5 (equal ALS/content weighting) and did not clearly outperform either individual model on ranking-hit metrics in this run. Tuning α, or exploring alternative blending strategies (e.g., rank-based fusion instead of score-based), is a natural next step.
- **Cold-start:** Neither the collaborative nor hybrid model was explicitly tested on true cold-start users (users with zero training history) — the content-based component provides a natural fallback path for this scenario that wasn't formally evaluated here.
- **Serverless compute constraints:** Several implementation decisions (manual top-K generation instead of `recommendForAllUsers`, avoiding `.cache()`/`.persist()`, avoiding RDD-based operations, Pandas-based flattening instead of Spark higher-order functions, materializing intermediate results to Parquet to avoid recomputation) were driven by Databricks serverless/Unity Catalog restrictions rather than algorithmic preference.



#### Conclusion

This project implemented and evaluated a Spark-based recommender system for the Sports & Outdoors category at scale (339,076 ratings, well above the 10k+ user/item threshold), comparing collaborative filtering (ALS), content-based filtering (TF-IDF over title, categories, and description), and a hybrid blend of both. The final ALS model achieved an RMSE of 1.6817 (rank=20, regParam=0.1, seed=42) — notably higher than the unseeded grid-search result for the same configuration (1.1371), directly illustrating ALS's sensitivity to random initialization.

Evaluated on a 1,000-user held-out sample across seven metrics, Content-Based filtering was the only model to register nonzero Precision@10, Recall@10, and NDCG@10, while also achieving the highest Coverage (8.42%) — at the cost of the lowest Diversity (0.1364), reflecting its tendency to recommend thematically narrow, similar items per user. ALS achieved the highest Diversity (0.7329) and the best (lowest) rescaled RMSE, having been the only model directly trained to predict ratings. The Hybrid model landed between the two on most metrics, illustrating both the benefit and the limits of a simple 50/50 candidate-blending approach.